<a href="https://colab.research.google.com/github/csanlag861-dev/BigData/blob/main/ejercicio_streaming.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Ejercicio patron de deteccion login.
## Stream(productor de eventos) que pasa por un wav que tiene que tener una ventana de tiempo.
## Si en la ventana de tiempo ha hecho login más de X veces, se envia al wav un baneo de ip por x minutos.
## El wav debe mantener una lista de usuarios baneados y su intervalo de tiempo.

In [1]:
import random
import time
from datetime import datetime

In [2]:
class StreamSimulator:
  """Simula un stream de eventos de login"""
  def __init__(self):
    self.usuarios = [f"user_{i}" for i in range(1,8)]
    # Escogemos solo un range de 1 a 8 para que se puedan repetir los usuarios.
    self.ips = {
        usuario: f"192.168.1.{random.randint(2, 254)}"
        for usuario in self.usuarios
        }

  def generar_usuario_aleatorio(self):
    return random.choice(self.usuarios)

  def generar_evento(self):
    """Generamos un evento aleatorio de login"""
    usuario_registrado = self.generar_usuario_aleatorio()
    return {
        'timestamp': datetime.now().isoformat(),
        'event_time': time.time(),
        'user_id': usuario_registrado,
        'ip_address': self.ips[usuario_registrado],
        'estado': random.choice(["SUCCESS", "FAILED"]),
        'dispositivo': random.choice(["Web", "Mobile", "API"])
    }

  def stream(self, n_eventos=10, delay_max=1.0):
    """Genera n eventos con delay aleatorio"""
    for i in range(n_eventos):
      evento = self.generar_evento()
      yield evento
      time.sleep(random.uniform(0.1, delay_max))

In [3]:
class AnomalyDetector:
  """Implementa una detección de anomalias"""
  def __init__(self, max_login_per_user=3, ban_duration=10, window_seconds=10):
    self.max_login_per_user = max_login_per_user
    self.ban_duration = ban_duration
    self.window_seconds = window_seconds
    self.usuarios_baneados = {}
    self.intentos = {}

  def detect_anomaly(self, event):
    """Detecta una anomalía en el login (ej: 3 intentos de login en 10 segundos)"""

    if event["estado"] != "FAILED":
      return False

    usuario = event["user_id"]
    timestamp = event["event_time"]
    ip = event["ip_address"]


    if usuario in self.usuarios_baneados:
      id_usuario_baneado = self.usuarios_baneados[usuario]
      tiempo_transcurrido = timestamp - id_usuario_baneado["timestamp_ban"]

      if tiempo_transcurrido >= id_usuario_baneado["duration"]:
        del self.usuarios_baneados[usuario]
        self.intentos[usuario] = []

        print(f"🔓 Baneo expirado: {usuario}")
      else:
        print(f"⛔ {usuario} sigue baneado. Evento ignorado.")
        return False

    if usuario not in self.intentos:
      self.intentos[usuario] = []


    self.intentos[usuario].append(timestamp)

    # Los intentos dentro de la ventana
    self.intentos[usuario] = [
      intento
      for intento in self.intentos[usuario]
      if timestamp - intento <= self.window_seconds
    ]

    print(
        f"Usuario: {usuario} | "
        f"Intentos en ventana: {len(self.intentos[usuario])}"
    )

    if len(self.intentos[usuario]) >= self.max_login_per_user:
        self.usuarios_baneados[usuario] = {
            "ip_address": ip,
            "timestamp_ban": timestamp,
            "duration": self.ban_duration
        }

        print(f"🚨 BANEADO: {usuario} | IP: {ip}")
        return True

    return False

In [4]:
simulator = StreamSimulator()
detector = AnomalyDetector()

print("=== SIMULACIÓN DE STREAM ===")

for i, evento in enumerate(
    simulator.stream(n_eventos=50, delay_max=0.5)
):
    detector.detect_anomaly(evento)

    print(f"\nEvento {i + 1}:")
    print(f"  Timestamp:    {evento['timestamp']}")
    print(f"  Usuario:      {evento['user_id']}")
    print(f"  Dirección IP: {evento['ip_address']}")
    print(f"  Estado:       {evento['estado']}")
    print(f"  Dispositivo:  {evento['dispositivo']}")

print("\n=== RESUMEN FINAL ===")
print(f"Usuarios baneados: {len(detector.usuarios_baneados)}")

for usuario, info in detector.usuarios_baneados.items():
    print(f"- {usuario} | IP: {info['ip_address']}")

=== SIMULACIÓN DE STREAM ===
Usuario: user_2 | Intentos en ventana: 1

Evento 1:
  Timestamp:    2026-11-02T20:39:50.234828
  Usuario:      user_2
  Dirección IP: 192.168.1.193
  Estado:       FAILED
  Dispositivo:  Web

Evento 2:
  Timestamp:    2026-11-02T20:39:50.587757
  Usuario:      user_7
  Dirección IP: 192.168.1.122
  Estado:       SUCCESS
  Dispositivo:  Mobile

Evento 3:
  Timestamp:    2026-11-02T20:39:51.010932
  Usuario:      user_7
  Dirección IP: 192.168.1.122
  Estado:       SUCCESS
  Dispositivo:  API

Evento 4:
  Timestamp:    2026-11-02T20:39:51.156369
  Usuario:      user_2
  Dirección IP: 192.168.1.193
  Estado:       SUCCESS
  Dispositivo:  Mobile
Usuario: user_6 | Intentos en ventana: 1

Evento 5:
  Timestamp:    2026-11-02T20:39:51.532312
  Usuario:      user_6
  Dirección IP: 192.168.1.60
  Estado:       FAILED
  Dispositivo:  Mobile
Usuario: user_6 | Intentos en ventana: 2

Evento 6:
  Timestamp:    2026-11-02T20:39:51.911545
  Usuario:      user_6
  Direcció